# CIS I — Assignment 1: Pivot Calibration

This notebook walks through the pivot-calibration scenario from the
assignment (Figure 1): a pointer and a calibration object, each
carrying a marker body, observed by an optical tracker. Your job is to
**design** the marker-sphere layouts and workspace placements, then
**calibrate** the pointer's tip position `p_tip` to within about 2mm.

Everything runs against `src.simulation`, a self-contained simulator that plays the role of the physical tracker +
hand-guided robot + calibration sensor described in the handout. It injects measurement noise — your code never sees ground truth
except where explicitly noted for **validation/reporting** in Step 8 (real
hardware wouldn't give you that; the point there is to report your achieved
accuracy, not to use the ground truth in your calibration math itself).



In [391]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))

import numpy as np
from src import simulation
from src import vct3, Rot, Frame

np.set_printoptions(precision=3, suppress=True)


## Step 1: Design the pointer's marker body

Choose approximate local marker-sphere positions `a_ptr,k` (a `List[vct3]`,
relative to the pointer's own coordinate frame) and the pointer tip's
approximate local position `p_tip_approx`. These are your
*nominal design values* — the simulator will independently perturb the
*actual* manufactured positions by a small, noise amount (bounded at
2mm).


In [413]:
ptr_local_positions = [
    # TODO: at least 4 non-coplanar vct3 points, e.g. vct3(x, y, z)
    # optimizing tracker placement here, ideally want to maximize volume and disrupl symmery
    # TODO: math proff/paper as backing
    vct3( 80,  80,  80),
    vct3( 80, -60, -80),
    vct3(-80,  65, -80),
    vct3(-70, -80,  70)
]
ptr_tip_approx = vct3(0, 0, -50)  # TODO: vct3, the tip's approximate position in the pointer's local frame


## Step 1.1: Design the calibration object's marker body

Same idea, for the calibration object's local marker positions `a_cal,k`.


In [393]:
cal_local_positions = [
    # TODO: at least 4 non-coplanar vct3 points
    # same idea here, avoiding symmetry
    vct3(-10, 35, -10),
    vct3(40, 40, 0),
    vct3(15, 10, 40),
    vct3(10, -20, 5)
]


## Step 2: Approximate workspace poses

Specify approximate poses (as `Frame` class from src) for the tracking camera
(`F_tracker`) and the calibration object (`F_cal`), relative to the
workspace. The tracker's usable volume is the tetrahedral region in the
handout's Figure 2 (roughly 100-200cm in front of the tracker, widening
from ~120cm to ~200cm across) — place the calibration object, and later
the pointer, so they're inside that volume.


In [394]:
F_tracker_approx = Frame(Rot(), vct3(1000, 1000, 2200))  # TODO: Frame, tracker pose relative to the workspace
F_cal_approx = Frame(Rot(), vct3(1200, 700, 400))      # TODO: Frame, calibration object pose relative to the workspace


## Step 3: Build the simulation

Fully provided — this just wires up your Step 1/2 design choices through
`simulation`'s `DefineMarkerBody` / `PlaceMarkerBody` / `CreateTracker`
equivalents. `ptr_placed`/`cal_placed` are `PlacedBody` handles: each pairs
a `MarkerBody` with its hidden actual placement, and gets passed to
`simulation.sample_marker_bodies(...)` whenever you want a reading.


In [415]:
ptr_mb = simulation.define_marker_body("ptr_mb", ptr_local_positions)
cal_mb = simulation.define_marker_body("cal_mb", cal_local_positions)

cal_placed = simulation.place_marker_body(cal_mb, F_cal_approx)
# The pointer's own body doesn't have a meaningful "workspace placement" yet
# It moves every time it's hand-guided (Step 4-6). 
# precision="precise" matches a robot-controlled pose rather than a coarsely, manually placed
# static object (see place_marker_body's docstring).
ptr_placed = simulation.place_marker_body(ptr_mb, Frame.eye(), precision="precise")

ptr = simulation.define_pointer("ptr1", ptr_tip_approx, ptr_mb)
trk = simulation.create_tracker("trk1", F_tracker_approx)

print("simulation built:", ptr_mb.name, cal_mb.name, ptr.name, trk.name)


simulation built: ptr_mb cal_mb ptr1 trk1


## Step 3.1: Refine `a_ptr,k` / `a_cal,k`

> "Observations of marker positions can be used to determine accurate
> estimates of relative positions a_ptr,k of markers on pointer and
> calibration objects... **Hint**: You probably want to take multiple
> readings in order to reduce the uncertainty associated with the a_k
> values."


In [416]:
def refine_marker_body(trk, placed, n_readings=100, passes=3):
    """
    Improve the estimated positions of a body's markers by averaging repeated tracker readings in the body's local coordinate frame
    
    input:
        trk (src.tracker.Tracker) - Tracker to read with
        placed (src.simulation.marker_body.PlacedBody) - PlacedBody whose body's nominal marker positions should be refined in place (via body.set_marker_positions(...))
        n_readings [int] - readings to average per pass
        passes[int] - how many times to repeat
    output: None (mutates placed.body's nominal_marker_positions in place)
    """
    if n_readings < 1:
        raise ValueError("n_readings must be >= 1")
    if passes < 1:
        raise ValueError("passes must be >= 1")
    if not isinstance(placed, simulation.marker_body.PlacedBody) or not isinstance(trk, simulation.tracker.Tracker):
        raise TypeError("placed must be a PlacedBody and trk must be a Tracker")
    if not placed.body.nominal_marker_positions:
        raise ValueError("placed.body must have nominal_marker_positions defined")
    
    #Go through 3 passes
    for pass_idx in range(passes):
        all_readings = []
        
        #Go through all the n_readings
        for _ in range(n_readings):
            
            #Gets the sample marker bodies and the observed markers and it's corresponding frame
            my_dict = simulation.sample_marker_bodies(trk, [placed])
            obs = my_dict[placed.body.name]
            frame = obs.frame
            
            #Have to get the inverse as we are going from the body to the tracker and not tracker to body
            to_body = frame.inv()
            
            local_positions = []
            reading = []
            
            #For each marker, find the local positions relative to the bodies' own positions.
            #If that is confusing think that to_body is the frame shift and p is the marker position. This finds each individual marker sphere!
            for marker in obs.markers:
                local_positions.append(to_body * marker.p)
            
            #Then get it in x,y,z so we can average it and then update
            for p in local_positions:
                reading.append ([p.x, p.y, p.z])
            
            all_readings.append(reading)
            
        #Get the averages of all the positions from all the passes and update the passes
        ave = np.mean(all_readings, axis=0)
        updated_positions = []
        for pos in ave:
            updated_positions.append(vct3(pos))

        placed.body.set_marker_positions(updated_positions)

        #tracking error over the passes
        actual_positions = placed.body.get_actual_marker_positions()
        mean_error = np.mean([
            (p_est - p_true.p).norm()
            for p_est, p_true in zip(updated_positions, actual_positions)
        ])
        
        print(f"Pass {pass_idx + 1}/{passes}: mean marker error = {mean_error:.4f} mm")
        

#initila error for ptr:
error = np.mean([
    (p_est - p_true.p).norm()
    for p_est, p_true in 
    zip(ptr_local_positions, ptr_placed.body.get_actual_marker_positions())])

print("Initial mean marker error after placement:", error, "mm")
refine_marker_body(trk, ptr_placed)
refine_marker_body(trk, cal_placed)

Initial mean marker error after placement: 0.9959522317480924 mm
Pass 1/3: mean marker error = 0.6481 mm
Pass 2/3: mean marker error = 0.6480 mm
Pass 3/3: mean marker error = 0.6480 mm
Pass 1/3: mean marker error = 1.0121 mm
Pass 2/3: mean marker error = 1.0121 mm
Pass 3/3: mean marker error = 1.0121 mm


## Steps 4-6: Collect pivot-calibration observations

For each observation `j`: hand-guide the pointer near the calibration
object at some desired orientation `R_desired,j`, take one simultaneous
tracker reading of *both* bodies, and read the calibration sensor. Loop
skeleton is provided; you choose how many observations to take and how to
vary `R_desired,j`


In [405]:
N_OBSERVATIONS = 64 # TODO: double check the number of observations

F_ptr_list = []
F_cal_list = []
sensed_tip_list = []
for j in range(N_OBSERVATIONS):
    R_desired = Rot.xyz((j // 8) * np.pi/28, 0, (j % 8) * np.pi/4)  #Gets 64 spherical combinations with x-tilt 0 to 45 and full azimuth sweep
    hg = simulation.hand_guide_pointer(ptr, ptr_placed, cal_placed, R_desired)
    obs = simulation.sample_marker_bodies(trk, [ptr_placed, cal_placed])
    sensed = simulation.sense_tip(ptr, hg)

    F_ptr_list.append(obs["ptr_mb"].frame)
    F_cal_list.append(obs["cal_mb"].frame)
    sensed_tip_list.append(sensed)

print(f"collected {len(F_ptr_list)} observations")


collected 64 observations


sensed_tip_list holds each observation's high-accuracy calibration-sensor reading p_sensed,j: the pointer tip's position relative to the calibration object's coordinate system, measured to negligible error (or None if that observation ended up outside the sensor's 5mm range). The hand-guided tip only lands approximately at the calibration object's origin, so the sensor tells you exactly where it actually was for each observation — an input to Step 7, together with F_ptr,j and F_cal,j.

## The Calibration Sensor (UPDATE)
The function p sensed = ReadCalibrationSensor() returns the actual value of the position of the pointer tip relative to the calibration sensor, so long as the tip is within 5 mm of the calibration sensor (i.e., ||p sensed|| ≤ 5). Otherwise it returns [−10000, −10000, −10000].

In [406]:
sensed_positions = []
valid_F_ptr_list = []
valid_F_cal_list = []

for index, sensed in enumerate(sensed_tip_list):
    #No variable taken
    if sensed is None:
        continue

    position = sensed.vec.ravel()

    # Error/Outside range
    if np.array_equal(position, [-10000, -10000, -10000]):
        continue

    sensed_positions.append(position)
    valid_F_ptr_list.append(F_ptr_list[index])
    valid_F_cal_list.append(F_cal_list[index])

#Find the covariance and at least 2 valid observations
if len(sensed_positions) >= 2:
    sensed_tip_cov = np.cov(
        np.array(sensed_positions), rowvar=False, ddof=1
    )
    
if sensed is None or np.array_equal(sensed.vec.ravel(), [-10000, -10000, -10000]):
    print(
        f"Observation {j}: sensor out of range | "
        f"x = {np.degrees((j // 8) * np.pi / 28):.2f}°, "
        f"z = {np.degrees((j % 8) * np.pi / 4):.2f}°"
    )
elif np.linalg.norm(sensed.vec) > 2:
    print(
        f"Observation {j}: "
        f"x = {np.degrees((j // 8) * np.pi / 28):.2f}°, "
        f"z = {np.degrees((j % 8) * np.pi / 4):.2f}° | "
        f"Distance = {np.linalg.norm(sensed.vec):.3f} mm"
    )

Observation 63: x = 45.00°, z = 315.00° | Distance = 3.185 mm


## Step 7: Compute a calibrated value for `p_tip`

This is the assignment's core deliverable — implement it yourself and specify the details of implementation in the report.

Each observation j gives you three measurements of the same physical moment: F_ptr,j and F_cal,j from the tracker, and p_sensed,j from the calibration sensor. Derive how they relate to the unknown p_tip, and use all three in your calibration.

Corner case to consider: What are the corner cases you want to analyse here? (e.g. what happens to observations whose p_sensed,j is None?)

Minimum data: how many observations, and what about their rotations, are needed for the stacked system to actually be solvable? How do you define whether a system is solvable here? Include that in your report


In [427]:
def compute_pivot_calibration(F_ptr_list, F_cal_list, sensed_positions):
    """
    Estimate the pointer's local tip position. Due to the tip being fixed relative to the pointer, 
    the transformed positions estimate the same point. We can then average them to obtain the calibrated tip position 
    
    input:
        F_ptr_list - List[Frame], the pointer marker body's pose relative to
            the tracker for each observation j (Fptr,j)
        F_cal_list - List[Frame], the calibration object's pose relative to
            the tracker for each observation j (Fcal,j), same order/length
        sensed_tip_list - List[Optional[vct3]], the calibration sensor's
            reading of the tip position relative to the calibration object
            for each observation j (p_sensed,j), same order/length; an entry
            is None if that observation's tip was out of sensor range
    output:
        p_tip - vct3, the calibrated tip position in the pointer's local frame
        cov_estimate - np.ndarray (3x3), an estimated covariance for p_tip
            (e.g. from the least-squares fit's residuals)
    """
    #Checks if the lists are the same length
    if (len(F_ptr_list) != len(F_cal_list) or len(F_cal_list) != len(sensed_positions)):
        raise ValueError("Lists are not the same length")
    
    #Checks if they are at least 3 observations as 2 observations could just be in the same plane
    if (len(F_ptr_list) < 2):
        raise ValueError("Lists aren't long enough and need more observations")
    
    tip_positions = []
    for index in range(len(sensed_positions)):
        #Get the sensor reading
        sensor_point = vct3(sensed_positions[index])
        
        #Move the point to tracker coords.
        tracker_point = F_cal_list[index] * sensor_point
        
        #Move to pointer coordinates
        pointer_point = F_ptr_list[index].inv() * tracker_point
        print(f"Observation {index}: pointer tip position estimate = {pointer_point.vec.ravel()}")
        
        #Store the 3 coordinates
        tip_positions.append(pointer_point.vec.ravel())
        
    #Average estimated tip position
    p_tip = vct3(np.mean(tip_positions, axis=0))
    
    #Cov estimated tip position
    cov_estimate = (
        np.cov(np.array(tip_positions), rowvar=False, ddof=1)/len(tip_positions)
    )
    
    return p_tip, cov_estimate
    
    

p_tip_est, p_tip_cov = compute_pivot_calibration(valid_F_ptr_list, valid_F_cal_list, sensed_positions)


Observation 0: pointer tip position estimate = [-0.469 -1.23  50.926]
Observation 1: pointer tip position estimate = [-0.478 -0.76  51.095]
Observation 2: pointer tip position estimate = [-0.068 -0.5   51.078]
Observation 3: pointer tip position estimate = [ 0.156 -0.522 51.074]
Observation 4: pointer tip position estimate = [ 0.607 -0.838 51.005]
Observation 5: pointer tip position estimate = [ 0.545 -1.316 51.063]
Observation 6: pointer tip position estimate = [ 0.213 -1.543 51.063]
Observation 7: pointer tip position estimate = [-0.29  -1.478 51.108]
Observation 8: pointer tip position estimate = [-0.548 -1.186 50.956]
Observation 9: pointer tip position estimate = [-0.434 -0.758 51.028]
Observation 10: pointer tip position estimate = [-0.115 -0.43  51.021]
Observation 11: pointer tip position estimate = [ 0.379 -0.493 51.016]
Observation 12: pointer tip position estimate = [ 0.581 -0.819 51.014]
Observation 13: pointer tip position estimate = [ 0.488 -1.304 51.02 ]
Observation 14: 

In [609]:
print("calibrated p_tip:", p_tip_est)
print("estimated covariance (mm^2):\n", p_tip_cov)
print("estimated std-dev per axis (mm):", np.sqrt(np.diag(p_tip_cov)))

calibrated p_tip: vct3(x=-0.17, y=0.24, z=50.15)
Internal Vector:
[[-0.171]
 [ 0.24 ]
 [50.155]]
estimated covariance (mm^2):
 [[ 0.005 -0.     0.   ]
 [-0.     0.005 -0.   ]
 [ 0.    -0.     0.001]]
estimated std-dev per axis (mm): [0.071 0.071 0.026]


## Step 8: Validate against test poses

> "**Hint**: You may want to consider placing additional marker bodies
> close to (but not inside) the cube defined by the test volume. However,
> this will also depend on your design for and placement of the
> calibration object."

The nominal test-pose grid is `theta_x, theta_y, theta_z` each in
`{0, +-pi/6, +-pi/3, +-pi/2}` and `p_x, p_y, p_z` each in
`{300, 300+-100, 300+-200}` mm. For each test pose, the assignment requires a
**single observation only** (no averaging).

Think about why the hint suggests an *additional*, nearby reference body?

In [610]:
near_local_positions = [
    vct3( 80,  320,  320),
    vct3( 80, -320, -320),
    vct3(-80,  320, -320),
    vct3(-80, -320,  320)
]
F_near_approx = Frame(Rot.xyz(0, 0, 0), vct3(650, 300, 300))

near_mb = simulation.define_marker_body("near_mb", near_local_positions)
near_placed = simulation.place_marker_body(near_mb, F_near_approx)
refine_marker_body(trk, near_placed)

Pass 1/3: mean marker error = 1.4622 mm
Pass 2/3: mean marker error = 1.4616 mm
Pass 3/3: mean marker error = 1.4613 mm


Refine this new body's geometry the same way you refined the pointer's and
calibration object's (Step 3b) before using it.


Now the single-observation test loop. For each nominal test pose `F_t`:
place the pointer there (small, robot-precision placement error, same as
the real system), take one simultaneous reading of the pointer and your
Step-8 reference body, and predict the tip's position (relative to that
reference body) using your calibrated `p_tip`.

**Ground truth for reporting only**: `PlacedBody.actual_placement` and
`MarkerBody.actual_marker_positions` hold the hidden values the simulator
sampled — you have access to them here only because this notebook doesn't
gate them behind a separate eval mode (per the handout: "During debugging
you will have access to all of the ... values. These will not be available
during evaluation."). Use them *only* to compute and report your achieved
error in this section — using them anywhere in your actual calibration math
(Steps 3b/7) kills the point of the assignment.


In [611]:
thetas = [0, np.pi / 6, -np.pi / 6, np.pi / 3, -np.pi / 3, np.pi / 2, -np.pi / 2]
offsets = [0, 100, -100, 200, -200]
# TODO: the full grid above is large (7^3 * 5^3 = 42,875 combinations) --
# choose a representative subset of (position, orientation) test poses
# rather than every combination.

#Building a list from the thetas and offsets above
test_poses = []
for theta in thetas:
    test_poses.append(Frame(Rot.xyz(theta, 0, 0), vct3(300, 300, 300)))
    for offset in offsets:
        test_poses.append(Frame(Rot.xyz(0, theta, 0), vct3(300+offset, 300, 300)))
        test_poses.append(Frame(Rot.xyz(0, 0, theta), vct3(300, 300+offset, 300)))
        test_poses.append(Frame(Rot.xyz(theta, theta, theta), vct3(300, 300, 300+offset)))

#Add corners for better representation
test_poses.append(Frame(Rot.xyz(0, 0, 0), vct3(100, 100, 100)))
test_poses.append(Frame(Rot.xyz(0, 0, 0), vct3(500, 500, 500)))

test_results = []  # list of (F_t_nominal, error_mm)
for F_t_nominal in test_poses:
    # TODO, for each F_t_nominal:
    #   1. test_placed = simulation.place_marker_body(ptr_mb, F_t_nominal, precision="precise")
    test_placed = simulation.place_marker_body(ptr_mb, F_t_nominal, precision="precise")
    
    #   2. obs = simulation.sample_marker_bodies(trk, [test_placed, near_placed])  -- single reading
    obs = simulation.sample_marker_bodies(trk, [test_placed, near_placed])
    
    #   3. predict the tip position relative to near_placed using p_tip_est
    pre_tip = obs[near_mb.name].frame.inv() * (obs[ptr_mb.name].frame * p_tip_est)
    
    #   4. compute the TRUE tip position relative to near_placed using
    #      test_placed.actual_placement / near_placed.actual_placement (ground truth,
    #      for reporting only -- see markdown above)    
    true_tip_w = test_placed.actual_placement * ptr.actual_tip_position.p
    true_tip_n = near_placed.actual_placement.inv() * true_tip_w

    #   5. record the error and append (F_t_nominal, error) to test_results
    error = np.linalg.norm((pre_tip - true_tip_n).vec)
    test_results.append((F_t_nominal, error))
    
    if error > 2:
        print("Position (mm):", F_t_nominal.p.vec.ravel())
        print("Rotation matrix:\n", F_t_nominal.R.matrix)
        print("Tip error (mm):", error)
    


Position (mm): [300. 400. 300.]
Rotation matrix:
 [[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]
Tip error (mm): 2.112160608523361
Position (mm): [300. 500. 300.]
Rotation matrix:
 [[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]
Tip error (mm): 2.2671799153876253
Position (mm): [100. 300. 300.]
Rotation matrix:
 [[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]
Tip error (mm): 2.0477132116992114
Position (mm): [300. 400. 300.]
Rotation matrix:
 [[ 0.866 -0.5    0.   ]
 [ 0.5    0.866  0.   ]
 [ 0.     0.     1.   ]]
Tip error (mm): 2.0477971199650584
Position (mm): [300. 500. 300.]
Rotation matrix:
 [[ 0.866 -0.5    0.   ]
 [ 0.5    0.866  0.   ]
 [ 0.     0.     1.   ]]
Tip error (mm): 2.2979992940907863
Position (mm): [300. 400. 300.]
Rotation matrix:
 [[ 0.866  0.5    0.   ]
 [-0.5    0.866  0.   ]
 [ 0.     0.     1.   ]]
Tip error (mm): 2.069753093694844
Position (mm): [200. 300. 300.]
Rotation matrix:
 [[ 0.866  0.    -0.5  ]
 [ 0.     1.     0.   ]
 [ 0.5    0.     0.866]]
Tip error (mm): 2.058899685912024
Posit

## Achieved performance

Summarize the tip-position error across the test-pose grid against the
handout's 2mm spec.


In [612]:
# TODO: summarize test_results -- e.g. mean/max error, and how many poses
# meet the assignment's ||delta p_t|| <= 2mm spec.

# TODO: average summary statistics for multiple runs?
errors = np.array([error for _, error in test_results])

print("Number of test poses:", len(errors))
print("Mean error (mm):", np.mean(errors))
print("Maximum error (mm):", np.max(errors))
print("Poses within 2 mm:", np.sum(errors <= 2), "/", len(errors))
print("Poses over 2 mm:", np.sum(errors > 2))
print("Standard deviation (mm):", np.std(errors, ddof=1))
print("Median error (mm):", np.median(errors))
print("95th percentile (mm):", np.percentile(errors, 95))
print("RMSE (mm):", np.sqrt(np.mean(errors ** 2)))

worst_index = np.argmax(errors)
print("Worst pose error (mm):", test_results[worst_index][1])

print()
errors_45 = np.array([
    error for F, error in test_results
    if F.R.matrix[2, 2] >= np.cos(np.pi / 4)
])
print("Number of test poses, with \npointer oriented within\n 45 degrees of vertical:", len(errors_45))
print("Mean error (mm):", np.mean(errors_45))
print("Maximum error (mm):", np.max(errors_45))
print("Poses within 2 mm:", np.sum(errors_45 <= 2), "/", len(errors_45))


Number of test poses: 114
Mean error (mm): 1.733312447378922
Maximum error (mm): 2.4110752123991297
Poses within 2 mm: 93 / 114
Poses over 2 mm: 21
Standard deviation (mm): 0.2854151488160816
Median error (mm): 1.778661189880696
95th percentile (mm): 2.2566845766266486
RMSE (mm): 1.7564507593885192
Worst pose error (mm): 2.4110752123991297

Number of test poses, with 
pointer oriented within
 45 degrees of vertical: 70
Mean error (mm): 1.8211026510500716
Maximum error (mm): 2.4110752123991297
Poses within 2 mm: 52 / 70


## UNIT TESTING FOR WRITTEN FUNCTIONS

In [ ]:
import unittest

class TestMarkerBodyRefinement(unittest.TestCase):

    def test_input(self):
        # Test with valid inputs
        trk = simulation.create_tracker("test_tracker", Frame.eye())
        placed = simulation.place_marker_body(ptr_mb, Frame.eye())
        try:
            refine_marker_body(trk, placed, n_readings=10, passes=2)
            refine_marker_body(trk, placed)  # default n_readings and passes
        except Exception as e:
            self.fail(f"refine_marker_body raised an exception with valid inputs: {e}")

        prt_empty_mb = simulation.define_marker_body("empty_mb", [])
        placed = simulation.place_marker_body(prt_empty_mb, Frame.eye())

        with self.assertRaises(ValueError):
            refine_marker_body(trk, placed)

        # Test with invalid tracker
        with self.assertRaises(TypeError):
            refine_marker_body(None, placed)

        # Test with invalid placed body
        with self.assertRaises(TypeError):
            refine_marker_body(trk, None)

        # Test with invalid n_readings
        with self.assertRaises(ValueError):
            refine_marker_body(trk, placed, n_readings=-1)

        # Test with another invalid n_readings
        with self.assertRaises(ValueError):
            refine_marker_body(trk, placed, n_readings=0)

        # Test with invalid passes
        with self.assertRaises(ValueError):
            refine_marker_body(trk, placed, passes=0)

        # Test with another invalid passes
        with self.assertRaises(ValueError):
            refine_marker_body(trk, placed, passes=-4)

    def TestOutput(self):
        #using initial tracker and pointer body initializations
        trk = simulation.create_tracker("test_tracker", F_tracker_approx)
        #making sure the test maker body is within the tracker range
        placed = simulation.place_marker_body(ptr_mb, Frame(Rot(), vct3(1000, 1000, 1000)))
        self.assertIsNone(refine_marker_body(trk, placed, n_readings=10, passes=2))

    def TestErrorReduction(self):
        #using initial tracker and pointer body initializations
        trk = simulation.create_tracker("test_tracker", F_tracker_approx)
        #making sure the test maker body is within the tracker range
        placed = simulation.place_marker_body(ptr_mb, Frame(Rot(), vct3(1000, 1000, 1000)))
        #get the initial error
        initial_error = np.mean([
            (p_est - p_true.p).norm()
            for p_est, p_true in 
            zip(ptr_local_positions, placed.body.get_actual_marker_positions())])
        
        refine_marker_body(trk, placed, n_readings=10, passes=2)
        
        #get the refined error
        refined_error = np.mean([
            (p_est - p_true.p).norm()
            for p_est, p_true in 
            zip(placed.body.nominal_marker_positions, placed.body.get_actual_marker_positions())])
        
        self.assertLess(refined_error, initial_error)

    def TestingNoErrorCase(self):
        #using initial tracker and pointer body initializations
        trk = simulation.create_tracker("test_tracker", F_tracker_approx)
        #making sure the test maker body is within the tracker range
        placed = simulation.place_marker_body(ptr_mb, Frame(Rot(), vct3(1000, 1000, 1000)))
        
        #set the nominal marker positions to be the same as the actual marker positions
        placed.body.set_marker_positions(placed.body.get_actual_marker_positions())
        
        #get the initial error
        initial_error = np.mean([
            (p_est - p_true.p).norm()
            for p_est, p_true in 
            zip(placed.body.nominal_marker_positions, placed.body.get_actual_marker_positions())])
        
        refine_marker_body(trk, placed, passes=1)
        
        #get the refined error
        refined_error = np.mean([
            (p_est - p_true.p).norm()
            for p_est, p_true in 
            zip(placed.body.nominal_marker_positions, placed.body.get_actual_marker_positions())])

        # We expect almost not change in the nominal marker positions
        for p_updated, p_actual in zip(placed.body.nominal_marker_positions, placed.body.get_actual_marker_positions()):
            self.assertAlmostEqual(p_updated.x, p_actual.p.x, places=3)
            self.assertAlmostEqual(p_updated.y, p_actual.p.y, places=3)
            self.assertAlmostEqual(p_updated.z, p_actual.p.z, places=3)
        # We expect that the is almost no change in the error
        self.assertAlmostEqual(refined_error, initial_error, places = 3)
        # We expect the refined error to be almost zero since the nominal marker positions are set to the actual marker positions
        self.assertAlmostEqual(refined_error, 0, places = 3)

In [402]:
# Run the tests
suite = unittest.TestLoader().loadTestsFromTestCase(TestMarkerBodyRefinement)
runner = unittest.TextTestRunner(verbosity=2)
runner.run(suite)

test_input (__main__.TestMarkerBodyRefinement.test_input) ... ok

----------------------------------------------------------------------
Ran 1 test in 0.154s

OK


Pass 1/2: mean marker error = 0.7468 mm
Pass 2/2: mean marker error = 0.7470 mm
Pass 1/3: mean marker error = 0.7473 mm
Pass 2/3: mean marker error = 0.7473 mm
Pass 3/3: mean marker error = 0.7472 mm


<unittest.runner.TextTestResult run=1 errors=0 failures=0>

In [ ]:
class TestPivotCalibration(unittest.TestCase):

    def test_input(self):
        # Test with valid inputs
        F_ptr_list = [Frame(Rot(), vct3(100, 100, 100)), Frame(Rot(), vct3(200, 200, 200))]
        F_cal_list = [Frame(Rot(), vct3(300, 300, 300)), Frame(Rot(), vct3(400, 400, 400))]
        sensed_positions = [vct3(10, 10, 10), vct3(20, 20, 20)]
        try:
            compute_pivot_calibration(F_ptr_list, F_cal_list, sensed_positions)
        except Exception as e:
            self.fail(f"compute_pivot_calibration raised an exception with valid inputs: {e}")

        # Test with invalid lengths
        with self.assertRaises(ValueError):
            compute_pivot_calibration(F_ptr_list, F_cal_list[:-1], sensed_positions)

        #Test with invalid lengths
        with self.assertRaises(ValueError):
            compute_pivot_calibration(F_ptr_list, F_cal_list, sensed_positions[:-1])

        # Test with insufficient observations
        with self.assertRaises(ValueError):
            compute_pivot_calibration(F_ptr_list[:1], F_cal_list[:1], sensed_positions[:1])

    def test_output(self):
        F_ptr_list = [Frame(Rot(), vct3(100, 100, 100)), Frame(Rot(), vct3(200, 200, 200))]
        F_cal_list = [Frame(Rot(), vct3(300, 300, 300)), Frame(Rot(), vct3(400, 400, 400))]
        sensed_positions = [vct3(10, 10, 10), vct3(20, 20, 20)]
        p_tip_est, p_tip_cov = compute_pivot_calibration(F_ptr_list, F_cal_list, sensed_positions)

        # test that the output types are correct
        self.assertIsInstance(p_tip_est, vct3)
        self.assertIsInstance(p_tip_cov, np.ndarray)
        # test that the covariance matrix has the correct shape
        self.assertEqual(p_tip_cov.shape, (3, 3))

    def test_tip_position_estimation(self):
        # Create a simple no error scenario with known transformations and sensed positions
        # Scenario 1, the pointer object is right above the calibration object at the z-axis and the pointer tip is at (0,0,0)
        # of the calibration coordinates.

        # Scenario 2, the pointer object has the tip at (0,0,0) of the calibration coordinates, but the pointer is rotated 90 degrees 
        # around the y-axis so that the tip is at is the calibration origin but the pointer object is "laying down" along the x-axis of the 
        # calibration object.
        F_ptr_list = [Frame(Rot(), vct3(0, 0, -2150)), Frame(Rot(axis='y', angle=np.pi/2), vct3(50, 0, -2200))]
        F_cal_list = [Frame(Rot(), vct3(0, 0, -2200)), Frame(Rot(), vct3(0, 0, -2200))]
        sensed_positions = [vct3(0, 0, 0), vct3(0, 0, 0)]

        p_tip_est, _ = compute_pivot_calibration(F_ptr_list, F_cal_list, sensed_positions)
        print("Estimated tip position:", p_tip_est)

        # The expected tip position in the pointer's local frame can be calculated manually
        expected_tip_position = vct3(0, 0, -50)  

        # Check that the estimated tip position is close to the expected position
        self.assertAlmostEqual(p_tip_est.x, expected_tip_position.x, places=2)
        self.assertAlmostEqual(p_tip_est.y, expected_tip_position.y, places=2)
        self.assertAlmostEqual(p_tip_est.z, expected_tip_position.z, places=2)


In [439]:
# Run the tests
suite = unittest.TestLoader().loadTestsFromTestCase(TestPivotCalibration)
runner = unittest.TextTestRunner(verbosity=2)
runner.run(suite)

test_input (__main__.TestPivotCalibration.test_input) ... ok
test_output (__main__.TestPivotCalibration.test_output) ... ok
test_tip_position_estimation (__main__.TestPivotCalibration.test_tip_position_estimation) ... ok

----------------------------------------------------------------------
Ran 3 tests in 0.015s

OK


Observation 0: pointer tip position estimate = [210. 210. 210.]
Observation 1: pointer tip position estimate = [220. 220. 220.]
Observation 0: pointer tip position estimate = [210. 210. 210.]
Observation 1: pointer tip position estimate = [220. 220. 220.]
Observation 0: pointer tip position estimate = [  0.   0. -50.]
Observation 1: pointer tip position estimate = [  0.   0. -50.]
Estimated tip position: vct3(x=0.00, y=0.00, z=-50.00)
Internal Vector:
[[  0.]
 [  0.]
 [-50.]]


<unittest.runner.TextTestResult run=3 errors=0 failures=0>